# AELIONIX BLACKFORGE — Phase 4 Colab Validation

This notebook performs a deterministic, one-click validation of the Blackforge **World Model Foundation** (Phase 4).

**What this validates:**
- Repository integrity and commit verification
- Dependency installation (runtime + dev extras)
- All Blackforge imports, including the new `blackforge.world_model` modules
- Full automated test suite (world model included)
- Bootstrap + health verification, including the new `world_model_ready` check
- Typed world model entities (13 kinds) with deterministic canonical identity and dedup
- Typed, directional, evidence-backed relationships (direction rules per type; symmetric dedup)
- **No fake authority**: OBSERVED/VALIDATED entities require evidence; the materializer floors status at the highest evidence status (HYPOTHESIZED when none)
- Evidence provenance in both directions (property-level and row-level)
- Corroboration raises confidence to the max, never lowers; no auto-increase from repetition
- Contradictions — weaker claims are recorded as assertions; the authoritative record is never silently overwritten
- Supersession — an OBSERVED/VALIDATED change supersedes the old record; full version history is preserved
- Mission isolation and session context on every query
- Bounded, deterministic neighborhood queries (no pathfinding)
- Restart persistence for entities, relationships, assertions and evidence links

**What this does NOT do:**
- No model download or LLM inference (Phase 4 is stdlib-only — SQLite + pydantic)
- No offensive security actions, reconnaissance, or scanning
- No autonomous attack planning
- No Attack Graph layer (LEADS_TO / ENABLES / EXPLOITS / CAN_COMPROMISE edges are rejected by design)

**Runtime:** Google Colab (CPU or GPU) — the notebook runs identically on free CPU runtimes.

---
## 1. Runtime Information

In [ ]:
import sys
import platform

print("Blackforge Phase 4 Colab Validation (World Model Foundation)")
print("=" * 60)
print("Python:", sys.version.split()[0])
print("Executable:", sys.executable)
print("Platform:", platform.platform())
print("Architecture:", platform.machine())
print("=" * 60)

assert sys.version_info >= (3, 10), f"Blackforge requires Python 3.10+, got {sys.version}"
print("Python version check: PASS")

---
## 2. Repository Acquisition

In [ ]:
from pathlib import Path
import subprocess

# ── Configuration (edit here if fork changes) ──────────────────────────
REPO_URL = "https://github.com/Sagelord00000001/Blackforge.git"
REPO_DIR = Path("/content/blackforge")
# ───────────────────────────────────────────────────────────────────────

if REPO_DIR.exists() and (REPO_DIR / "blackforge" / "__init__.py").exists():
    print(f"Repository already exists at {REPO_DIR}, updating...")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
else:
    if REPO_DIR.exists():
        import shutil
        shutil.rmtree(REPO_DIR)
    subprocess.run(
        ["git", "clone", REPO_URL, str(REPO_DIR)],
        check=True,
    )

import os
os.chdir(str(REPO_DIR))
print(f"Repository ready at {REPO_DIR}")

---
## 3. Commit Verification

In [ ]:
import subprocess

EXPECTED_PHASE3_COMMIT = "9041d57"

result = subprocess.run(
    ["git", "-C", str(REPO_DIR), "log", "-1", "--format=%h"],
    capture_output=True, text=True, check=True,
)
current_commit = result.stdout.strip()

print(f"Expected Phase 3 baseline: {EXPECTED_PHASE3_COMMIT}")
print(f"Current repository commit: {current_commit}")

result_log = subprocess.run(
    ["git", "-C", str(REPO_DIR), "log", "--oneline"],
    capture_output=True, text=True, check=True,
)
commits = [line.split()[0] for line in result_log.stdout.strip().splitlines()]

has_world_model = (
    (REPO_DIR / "blackforge" / "world_model" / "repository.py").exists()
    and (REPO_DIR / "blackforge" / "world_model" / "store.py").exists()
)
has_phase3_evidence = (REPO_DIR / "blackforge" / "evidence" / "bridge.py").exists()

if not has_world_model:
    raise RuntimeError("Phase 4 world model modules not present — repo is ahead or behind.")
print("Phase 4 world model modules present: PASS")
if not has_phase3_evidence:
    raise RuntimeError("Phase 3 evidence modules missing unexpectedly.")
print("Phase 3 evidence modules present: PASS")

if EXPECTED_PHASE3_COMMIT in commits:
    print("Commit verification: PASS (Phase 3 baseline found)")
elif any(c.startswith(EXPECTED_PHASE3_COMMIT[:4]) for c in commits):
    print("Commit verification: PASS (Phase 3 baseline found, abbreviated match)")
else:
    result_merge = subprocess.run(
        ["git", "-C", str(REPO_DIR), "merge-base", "--is-ancestor",
         EXPECTED_PHASE3_COMMIT, current_commit],
        capture_output=True, check=False,
    )
    if result_merge.returncode == 0:
        print("Commit verification: PASS (repo advanced past Phase 3)")
    else:
        raise RuntimeError("Phase 3 commit not found in repository history.")
print("Commit verification: PASS")

---
## 4. Install Blackforge

Phase 4 (world model) uses only the Python standard library plus the existing runtime (SQLite + pydantic). Only the `[dev]` extra is installed — the `llm` extra (torch/transformers) is intentionally left out, so this notebook is lightweight and OOM-free on free CPU runtimes.

In [ ]:
!pip install hatchling --quiet
!pip install -e ".[dev]"

import blackforge
print("Blackforge import: PASS")

---
## 5. Environment / Import Health Check

In [ ]:
import importlib

modules = [
    "blackforge",
    "blackforge.core.config",
    "blackforge.core.errors",
    "blackforge.core.types",
    "blackforge.runtime.bootstrap",
    "blackforge.memory",
    "blackforge.evidence",
    "blackforge.world_model",
    "blackforge.world_model.models",
    "blackforge.world_model.canonical",
    "blackforge.world_model.rules",
    "blackforge.world_model.query",
    "blackforge.world_model.repository",
    "blackforge.world_model.store",
    "blackforge.world_model.materializer",
    "blackforge.mission.manager",
    "blackforge.capabilities.registry",
    "blackforge.authorization",
    "blackforge.scope.validator",
]

_import_failures = []
for module in modules:
    try:
        importlib.import_module(module)
    except Exception as e:
        _import_failures.append((module, str(e)))

if _import_failures:
    for mod, err in _import_failures:
        print(f"  FAIL: {mod} — {err}")
    raise RuntimeError(f"Import health check failed: {len(_import_failures)} module(s)")

print(f"Blackforge imports OK ({len(modules)} modules verified).")
print("World model module imports: PASS")

---
## 6. Automated Regression Tests

Runs the full suite including `tests/test_world_model_phase4.py`. The LLM/torch-heavy files are excluded: importing the HF provider pulls ~2GB of torch memory and can SIGKILL the kernel on CPU runtimes.

In [ ]:
import subprocess
import sys

print("Running automated test suite...")
result = subprocess.run(
    [
        sys.executable, "-m", "pytest", "-q", "--tb=short",
        "--ignore=tests/test_huggingface_provider.py",
        "--ignore=tests/test_loader.py",
        "--ignore=tests/test_smoke_real_model.py",
    ],
    capture_output=True, text=True, cwd=str(REPO_DIR),
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-500:] if result.stderr else "")
    raise RuntimeError(f"pytest failed with exit code {result.returncode}")

print("Automated test suite: PASS")

---
## 7. Bootstrap Verification

Bootstraps the app and confirms every subsystem — including the new `world_model_ready` — reports healthy against a disposable database in the Colab working directory.

In [ ]:
import os
from pathlib import Path

os.environ["BLACKFORGE_WORLD_MODEL_DB_PATH"] = str(Path("data/phase4_colab_world_model.db").resolve())
Path("data").mkdir(exist_ok=True)
Path("data/phase4_colab_world_model.db").unlink(missing_ok=True)

from blackforge.runtime.bootstrap import bootstrap

app = bootstrap()
assert app.healthy(), "Blackforge health check failed"
verification = app.verify()
assert verification["evidence_store_ready"], "evidence_store_ready must be True"
assert verification["evidence_memory_link_ready"], "evidence_memory_link_ready must be True"
assert verification["memory_ready"], "memory_ready must be True"
assert verification["world_model_ready"], "world_model_ready must be True"

BOOTSTRAP_OK = app.healthy() and verification["world_model_ready"]  # snapshot before cleanup

for k, v in verification.items():
    symbol = "PASS" if v else "FAIL"
    print(f"  [{symbol}] {k}")

print("\nBlackforge bootstrap (world model ready): PASS")

---
## 8. Entity Identity & Deterministic Dedup

Entities are keyed by a deterministic canonical identity scoped to `(mission, entity_type, namespace, normalized_name)`. Internal IDs are never the dedup basis. Distinct-but-similar names are never merged; URL/hostname/network names are normalized cautiously (case, trailing dots, default ports).

In [ ]:
from blackforge.world_model import (
    EntitySpec, RelationshipSpec, EntityType, RelationshipType,
    EvidenceLinkRef, WorldModelStore,
)
from blackforge.world_model.models import WorldMutation, WorldLifecycle
from blackforge.world_model.repository import InMemoryWorldRepository
from blackforge.world_model.query import WorldQuery
from blackforge.core.types import (
    MissionID, SessionID, EvidenceID, Confidence, EvidenceStatus,
)
from blackforge.core.errors import WorldRuleError

MID = MissionID("mission_phase4")
SID = SessionID("sess_phase4")
SID_OTHER = SessionID("sess_phase4_b")

wm = WorldModelStore(InMemoryWorldRepository())  # disposable in-memory store
assert wm.health_check()
print("World model health check: PASS")

def entity(name, etype=EntityType.ENDPOINT, **over):
    base = dict(
        mission_id=MID, session_id=SID, entity_type=etype, name=name,
        epistemic_status="observed", confidence=Confidence.HIGH,
        evidence=[EvidenceLinkRef(evidence_id=EvidenceID(f"ev_{abs(hash(name)) % 10**9}"))],
    )
    base.update(over)
    return EntitySpec(**base)

# 1) Same canonical identity dedups to the same internal record (CORROBORATED).
a1 = wm.add_entity(entity("HTTPS://WEB.EXAMPLE.COM", properties={"port": 443}))
a2 = wm.add_entity(entity("https://web.example.com/", properties={"port": 443}))
assert a1.action == WorldMutation.CREATED
assert a2.action == WorldMutation.CORROBORATED
assert a2.entity.id == a1.entity.id
assert a2.entity.version == 1
print(f"Same canonical identity -> same record {a1.entity.id}: PASS")

# 2) Canonical key is deterministic and mission-scoped.
canon = a1.entity.canonical_key
assert canon == "endpoint||https://web.example.com/"
assert wm.find_entity(MID, EntityType.ENDPOINT, "https://web.example.com").canonical_key == canon
print("Deterministic canonical key (readable, not hashed): PASS")

# 3) Distinct-but-similar names are never merged.
one = wm.add_entity(entity("web.example.com", EntityType.SERVICE))
two = wm.add_entity(entity("web-1.example.com", EntityType.SERVICE))
assert one.entity.id != two.entity.id
assert wm.count_entities(MID) >= 3
print("Distinct-but-similar names stay distinct: PASS")

# 4) Namespaces scope identity.
dev = wm.add_entity(entity("api", EntityType.SERVICE, namespace="dev"))
prod = wm.add_entity(entity("api", EntityType.SERVICE, namespace="prod"))
assert dev.entity.id != prod.entity.id
print("Namespace-scoped identity: PASS")

# 5) No fake authority: OBSERVED without evidence is rejected.
rejected = False
try:
    wm.add_entity(EntitySpec(mission_id=MID, session_id=SID,
                             entity_type=EntityType.APPLICATION, name="sneaky",
                             epistemic_status="observed", evidence=[]))
except WorldRuleError:
    rejected = True
assert rejected
print("Authoritative status without evidence rejected: PASS")

print("\nWorld model identity & dedup: PASS")

---
## 9. Relationships, Direction Rules & Evidence Provenance

Two endpoints must exist, be ACTIVE, and belong to the SAME mission. Symmetric types (CONNECTS_TO, ASSOCIATED_WITH) dedup order-insensitively; directed types keep direction. Evidence is linked in both directions and merged on corroboration.

In [ ]:
def endpoint(name):
    return wm.add_entity(entity(name))

svc_a = wm.add_entity(entity("svc-a", EntityType.SERVICE))
svc_b = wm.add_entity(entity("svc-b", EntityType.SERVICE))

# 1) Directed edge: A->B and B->A are distinct records.
fwd = wm.add_relationship(RelationshipSpec(
    mission_id=MID, session_id=SID, relationship_type=RelationshipType.TRUSTS,
    source_entity_id=svc_a.entity.id, target_entity_id=svc_b.entity.id,
    evidence=[EvidenceLinkRef(evidence_id=EvidenceID("ev_tr1"))],
))
rev = wm.add_relationship(RelationshipSpec(
    mission_id=MID, session_id=SID, relationship_type=RelationshipType.TRUSTS,
    source_entity_id=svc_b.entity.id, target_entity_id=svc_a.entity.id,
    evidence=[EvidenceLinkRef(evidence_id=EvidenceID("ev_tr2"))],
))
assert fwd.relationship.id != rev.relationship.id
print("Directed edges keep direction (A->B != B->A): PASS")

# 2) Symmetric edge: A-B and B-A dedup to the same record.
sab = wm.add_relationship(RelationshipSpec(
    mission_id=MID, session_id=SID, relationship_type=RelationshipType.CONNECTS_TO,
    source_entity_id=svc_a.entity.id, target_entity_id=svc_b.entity.id,
    evidence=[EvidenceLinkRef(evidence_id=EvidenceID("ev_c1"))],
))
sba = wm.add_relationship(RelationshipSpec(
    mission_id=MID, session_id=SID, relationship_type=RelationshipType.CONNECTS_TO,
    source_entity_id=svc_b.entity.id, target_entity_id=svc_a.entity.id,
    evidence=[EvidenceLinkRef(evidence_id=EvidenceID("ev_c2"))],
))
assert sba.action == WorldMutation.CORROBORATED
assert sba.relationship.id == sab.relationship.id
print("Symmetric edges dedup order-insensitively: PASS")

# 3) Self-loops and cross-mission endpoints are rejected.
rejected = False
try:
    wm.add_relationship(RelationshipSpec(mission_id=MID, session_id=SID,
                                         relationship_type=RelationshipType.TRUSTS,
                                         source_entity_id=svc_a.entity.id,
                                         target_entity_id=svc_a.entity.id))
except WorldRuleError:
    rejected = True
assert rejected
other_wm_endpoint = wm.add_entity(entity("https://outside.example.com", mission_id=MissionID("other")))
rejected_cross = False
try:
    wm.add_relationship(RelationshipSpec(mission_id=MID, session_id=SID,
                                         relationship_type=RelationshipType.USES,
                                         source_entity_id=svc_a.entity.id,
                                         target_entity_id=other_wm_endpoint.entity.id))
except WorldRuleError:
    rejected_cross = True
assert rejected_cross
print("Self-loop and cross-mission edges rejected: PASS")

# 4) Evidence provenance, both directions.
rels_of = wm.list_relationships(__import__("blackforge.world_model.query", fromlist=["RelationshipQuery"]).RelationshipQuery(mission_id=MID))
ids_ab = {str(r.id) for r in rels_of if str(r.source_entity_id) == str(svc_a.entity.id) and str(r.target_entity_id) == str(svc_b.entity.id)}
links = wm.evidence_for_relationship(str(sab.relationship.id))
assert {link["evidence_id"] for link in links} == {"ev_c1", "ev_c2"}
print("Evidence merged on corroboration (2 refs on 1 relationship): PASS")

entity_ev = wm.evidence_for_entity(str(svc_a.entity.id))
assert entity_ev and isinstance(entity_ev[0]["evidence_id"], str)
print(f"Evidence on entity is non-empty and typed ({len(entity_ev)} ref(s)): PASS")

print("\nWorld model relationships & provenance: PASS")

---
## 10. No-Fake-Authority Materializer & Confidence

The materializer is deterministic and evidence-driven: an entity's epistemic status never exceeds the highest status of its linked evidence (and is HYPOTHESIZED with none). Corroboration raises confidence to the maximum, never lowers it, and repetition alone never increases it.

In [ ]:
from blackforge.world_model.materializer import WorldMaterializer, EntityFact

mat = WorldMaterializer(wm)

# 1) No evidence status -> HYPOTHESIZED floor (never authority).
gw = mat.materialize_entity(
    MID,
    EntityFact(entity_type=EntityType.APPLICATION, name="api-gateway",
               evidence=[EvidenceLinkRef(evidence_id=EvidenceID("ev_llm_claim"))]),
    evidence_statuses=[],
)
assert gw.entity.epistemic_status == EvidenceStatus.HYPOTHESIZED
print("Materializer floors status at HYPOTHESIZED without evidence: PASS")

# 2) Observed evidence raises the floor.
obs = mat.materialize_entity(
    MID,
    EntityFact(entity_type=EntityType.APPLICATION, name="billing-app",
               evidence=[EvidenceLinkRef(evidence_id=EvidenceID("ev_scan"))]),
    evidence_statuses=[EvidenceStatus.HYPOTHESIZED, EvidenceStatus.OBSERVED],
)
assert obs.entity.epistemic_status == EvidenceStatus.OBSERVED
print("Materializer inherits highest evidence status (OBSERVED): PASS")

# 3) Confidence: corroboration raises to max, never lowers.
c1 = wm.add_entity(entity("conf.example.com", EntityType.SERVICE, confidence=Confidence.LOW))
c2 = wm.add_entity(entity("conf.example.com", EntityType.SERVICE, confidence=Confidence.CONFIRMED))
assert c2.action == WorldMutation.CORROBORATED
assert c2.entity.confidence == Confidence.CONFIRMED
c3 = wm.add_entity(entity("conf.example.com", EntityType.SERVICE, confidence=Confidence.LOW))
assert c3.entity.confidence == Confidence.CONFIRMED
print("Confidence raised to max, never lowered: PASS")

print("\nMaterializer & confidence rules: PASS")

---
## 11. Contradiction & Supersession History

A weaker (HYPOTHESIZED/INFERRED) claim that conflicts with the authoritative record is stored as an assertion bound to the entity — nothing is silently overwritten. An OBSERVED/VALIDATED change supersedes the previous version, which stays visible as history.

In [ ]:
# 1) Weak contradiction -> assertion recorded, record untouched.
web = wm.add_entity(entity("https://web.example.com", properties={"port": 443}))
weak = wm.add_entity(entity(
    "https://web.example.com",
    properties={"port": 8080},
    epistemic_status="hypothesized",
    confidence=Confidence.LOW, evidence=[],
))
assert weak.action == WorldMutation.CONTRADICTION_RECORDED
assert weak.entity.properties == {"port": 443}, "authoritative record must be untouched"
assert weak.assertion is not None
assert weak.assertion.property_key == "port" and weak.assertion.property_value == "8080"
assert len(wm.list_assertions(str(web.entity.id))) == 1
print("Weak contradiction stored as assertion, record untouched: PASS")

# 2) Authoritative supersession preserves version history.
v1 = wm.add_entity(entity("https://legacy.example.com", properties={"version": "1"}))
v2 = wm.add_entity(entity("https://legacy.example.com", properties={"version": "2"}))
assert v2.action == WorldMutation.SUPERSEDED
assert v2.entity.version == 2
assert str(v2.entity.supersedes) == str(v1.entity.id)
old = wm.get_entity(str(v1.entity.id))
assert old.lifecycle == WorldLifecycle.SUPERSEDED
assert old.properties == {"version": "1"}
new = wm.get_entity(str(v2.entity.id))
assert new.lifecycle == WorldLifecycle.ACTIVE and new.properties == {"version": "2"}
assert wm.count_entities(MID) >= 2  # history retained, nothing deleted
print("Supersession keeps full history (version 2, supersedes link): PASS")

# 3) INFERRED disagreement NEVER supersedes.
weak2 = wm.add_entity(entity(
    "https://legacy.example.com",
    properties={"version": "9"},
    epistemic_status="inferred", confidence=Confidence.MEDIUM,
    evidence=[EvidenceLinkRef(evidence_id=EvidenceID("ev_inferred"))],
))
assert weak2.action == WorldMutation.CONTRADICTION_RECORDED
cur = wm.get_entity(str(v2.entity.id))
assert cur.properties == {"version": "2"}
print("Inferred disagreement never overwrites authority: PASS")

print("\nContradiction & supersession: PASS")

---
## 12. Mission Isolation, Session Context & Neighborhood Query

Every world model operation is mission-scoped; cross-mission reads are impossible. Session is an optional narrowing filter. Neighborhood queries are bounded (depth <= 2) and deterministic — no pathfinding, ever.

In [ ]:
# 1) Same identity in another mission is a distinct record.
mid_other = MissionID("mission_phase4_other")
wm.add_entity(entity("https://web.example.com", mission_id=mid_other, session_id=SID_OTHER))
assert wm.count_entities(MID) != wm.count_entities(mid_other)
same_in_other = wm.find_entity(mid_other, EntityType.ENDPOINT, "https://web.example.com")
assert same_in_other is not None and same_in_other.canonical_key == canon
print("Mission isolation (same identity, distinct mission): PASS")

# 2) Session context narrows within a mission.
n_sess1 = wm.count_entities(MID, session_id=SID)
n_sess2 = wm.count_entities(MID, session_id=SID_OTHER)
wm.add_entity(entity("sessioned.example.com", EntityType.SERVICE, session_id=SID_OTHER))
assert wm.count_entities(MID, session_id=SID_OTHER) == n_sess2 + 1
assert wm.count_entities(MID, session_id=SID) == n_sess1
print("Session context filters within a mission: PASS")

# 3) Bounded, deterministic neighborhood (up to depth 2).
hub = wm.add_entity(entity("hub.example.com", EntityType.SERVICE))
leaf = wm.add_entity(entity("leaf.example.com", EntityType.SERVICE))
spoke = wm.add_entity(entity("spoke.example.com", EntityType.SERVICE))
wm.add_relationship(RelationshipSpec(mission_id=MID, session_id=SID,
                                    relationship_type=RelationshipType.DEPENDS_ON,
                                    source_entity_id=hub.entity.id,
                                    target_entity_id=leaf.entity.id))
wm.add_relationship(RelationshipSpec(mission_id=MID, session_id=SID,
                                    relationship_type=RelationshipType.DEPENDS_ON,
                                    source_entity_id=hub.entity.id,
                                    target_entity_id=spoke.entity.id))
n1 = wm.neighborhood(str(hub.entity.id), direction="out", max_depth=3)
assert n1 is not None
assert n1.depth == 1, "depth is bounded, no pathfinding"
assert {str(e.id) for e in n1.entities} == {str(leaf.entity.id), str(spoke.entity.id)}
n2 = wm.neighborhood(str(hub.entity.id), direction="out", max_depth=3)
assert [str(r.id) for r in n1.relationships] == [str(r.id) for r in n2.relationships]
assert n2.neighborhood_entities if hasattr(n2, "neighborhood_entities") else True
print(f"Neighborhood bounded (depth=1, {len(n1.relationships)} edges), deterministic order: PASS")

print("\nMission/session isolation & neighborhood: PASS")

---
## 13. Restart Persistence

Entities, relationships, assertions and evidence links survive a full close/reopen on brand-new SQLite connections — the same guarantee as Phase 2/3.

In [ ]:
from pathlib import Path
from blackforge.world_model.repository import SQLiteWorldRepository

Path("data").mkdir(exist_ok=True)
WM_DB = str(Path("data/phase4_colab_world_model.db"))
Path(WM_DB).unlink(missing_ok=True)

# ── Writer instances ────────────────────────────────────────────────────
w_wm = WorldModelStore(SQLiteWorldRepository(WM_DB))
w_entity = w_wm.add_entity(entity("persist.example.com", EntityType.SERVICE))
w_pair = (w_wm.add_entity(entity("persist-a.example.com", EntityType.SERVICE)),
          w_wm.add_entity(entity("persist-b.example.com", EntityType.SERVICE)))
w_rel = w_wm.add_relationship(RelationshipSpec(
    mission_id=MID, session_id=SID,
    relationship_type=RelationshipType.USES,
    source_entity_id=w_pair[0].entity.id,
    target_entity_id=w_pair[1].entity.id,
    evidence=[EvidenceLinkRef(evidence_id=EvidenceID("ev_persist"))],
))
w_assert = w_wm.add_assertion(__import__(
    "blackforge.world_model.models", fromlist=["AssertionSpec"]
).AssertionSpec(
    mission_id=MID, session_id=SID, entity_id=w_entity.entity.id,
    property_key="role", property_value="primary",
    epistemic_status="hypothesized",
))
w_wm.close()
print("Writer closed. World model records persisted.")

# ── Reader instances (brand new connections) ────────────────────────────
r_wm = WorldModelStore(SQLiteWorldRepository(WM_DB))
persisted = r_wm.find_entity(MID, EntityType.SERVICE, "persist.example.com")
assert persisted is not None and persisted.canonical_key == "service||persist.example.com"
rel_loaded = r_wm.get_relationship(str(w_rel.relationship.id))
assert rel_loaded is not None
assert {ref["evidence_id"] for ref in r_wm.evidence_for_relationship(str(w_rel.relationship.id))} == {"ev_persist"}
assert len(r_wm.list_assertions(str(w_entity.entity.id))) == 1
print("Restart persistence (entities, relationships, evidence, assertions): PASS")

---
## 14. Transaction / Rule Integrity

Rule violations leave no partial writes, and a failed transaction rolls back atomically. Health probes are self-cleaning — they never leave records behind.

In [ ]:
_ = wm.count_entities(MID)

# Failed rule check leaves no partial relationship.
before_edges = len(wm.list_relationships(__import__(
    "blackforge.world_model.query", fromlist=["RelationshipQuery"]
).RelationshipQuery(mission_id=MID)))
rejected = False
try:
    wm.add_relationship(RelationshipSpec(mission_id=MID, session_id=SID,
                                         relationship_type=RelationshipType.USES,
                                         source_entity_id="went_0000000000000000",
                                         target_entity_id="went_0000000000000000"))
except WorldRuleError:
    rejected = True
assert rejected
assert len(wm.list_relationships(__import__(
    "blackforge.world_model.query", fromlist=["RelationshipQuery"]
).RelationshipQuery(mission_id=MID))) == before_edges
print("Rule failure leaves no partial state: PASS")

# Health probe is self-cleaning.
assert wm.health_check()
assert len(wm.list_entities(WorldQuery(mission_id=MissionID("health_check")))) == 0
print("Health probe leaves no residue: PASS")

print("\nTransaction / rule integrity: PASS")

---
## 15. Clean Up

Close every open backend and remove the disposable database.

In [ ]:
for store in (wm, r_wm):
    try:
        store.close()
    except Exception:
        pass
try:
    app.world_model.close()
except Exception:
    pass
Path("data/phase4_colab_world_model.db").unlink(missing_ok=True)
print("Backends closed and disposable database removed. Validation complete.")

---
## 16. PASS/FAIL Summary

In [ ]:
RESULTS = {}

RESULTS["repository"] = (REPO_DIR / "blackforge" / "__init__.py").exists()
RESULTS["phase4_modules"] = ((REPO_DIR / "blackforge" / "world_model" / "store.py").exists()
                             and (REPO_DIR / "blackforge" / "world_model" / "materializer.py").exists())
RESULTS["imports"] = len(_import_failures) == 0
RESULTS["tests"] = True  # Would have raised if failed
RESULTS["bootstrap"] = BOOTSTRAP_OK
RESULTS["identity_dedup"] = (a2.action == WorldMutation.CORROBORATED
                             and a2.entity.id == a1.entity.id
                             and one.entity.id != two.entity.id)
RESULTS["no_fake_authority"] = (rejected
                                and gw.entity.epistemic_status == EvidenceStatus.HYPOTHESIZED
                                and obs.entity.epistemic_status == EvidenceStatus.OBSERVED)
RESULTS["confidence"] = (c2.entity.confidence == Confidence.CONFIRMED
                         and c3.entity.confidence == Confidence.CONFIRMED)
RESULTS["direction_rules"] = (fwd.relationship.id != rev.relationship.id
                              and sab.relationship.id == sba.relationship.id)
RESULTS["provenance"] = {link["evidence_id"] for link in wm.evidence_for_relationship(str(sab.relationship.id))} == {"ev_c1", "ev_c2"}
RESULTS["contradiction"] = (weak.action == WorldMutation.CONTRADICTION_RECORDED
                            and weak.assertion is not None
                            and weak.entity.properties == {"port": 443})
RESULTS["supersession"] = (v2.entity.version == 2
                           and str(v2.entity.supersedes) == str(v1.entity.id)
                           and old.lifecycle == WorldLifecycle.SUPERSEDED
                           and new.lifecycle == WorldLifecycle.ACTIVE)
RESULTS["mission_isolation"] = same_in_other is not None and wm.count_entities(MID) >= 1
RESULTS["neighborhood"] = (n1 is not None and n1.depth == 1
                           and {str(e.id) for e in n1.entities} == {str(leaf.entity.id), str(spoke.entity.id)}
                           and [str(r.id) for r in n1.relationships] == [str(r.id) for r in n2.relationships])
RESULTS["restart_persistence"] = (persisted is not None and rel_loaded is not None)

passed = sum(1 for v in RESULTS.values() if v)
failed = sum(1 for v in RESULTS.values() if not v)

print("PHASE 4 VALIDATION SUMMARY")
print("=" * 60)
for key, value in RESULTS.items():
    symbol = "PASS" if value else "FAIL"
    print(f"  [{symbol}] {key}")
print("=" * 60)
print(f"RESULT: {passed} passed, {failed} failed")
if failed:
    raise RuntimeError(f"Phase 4 validation failed: {failed} check(s) failed")

print("\nPHASE 4 VALIDATION: OVERALL PASS")